<div align="center">

# 🚕 Yandex Go Analytics

## 2 · Batch Ingestion

*Read Parquet and Avro files straight from the Volume and persist them in Bronze — as they are.*

![layer](https://img.shields.io/badge/layer-bronze-CD7F32)
![ingestion](https://img.shields.io/badge/ingestion-Auto%20Loader-FF3621)
![formats](https://img.shields.io/badge/formats-Parquet%20%2B%20Avro-00ADD4)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [1 · Setup](./1.%20Setup.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb) |

**Progress** &nbsp; ▰▰▱▱▱▱▱▱  **2 / 8**

---

## 🎯 Task requirements

- [x] Read the **`.parquet`** and **`.avro`** files directly from the Volume
- [x] Write them **"as-is"** into the `bronze` schema as **Delta** tables: `bronze.taxi` and `bronze.users`

## 🧩 How it is solved

Both tables use the same service, `BaseIngestionService`:

| Step | What happens |
|:--|:--|
| 1 · **Extract** | Auto Loader (`cloudFiles`) reads new files from the Volume; the schema is inferred and stored next to the checkpoint |
| 2 · **Audit columns** | `_ingested_at` (timestamp) and `_source_file` (path) are added — the business columns stay untouched |
| 3 · **Load** | Append to a Delta table with `trigger(availableNow=True)`: process everything available, then stop |

| Entity | Source | Target |
|:--|:--|:--|
| 🚕 Trips | `…/landing/parquet` | `bronze.taxi` |
| 👤 Users | `…/landing/avro` | `bronze.users` |

> 💡 **Incremental by design** — the checkpoint remembers which files were processed, so a re-run ingests only
> *new* files and never duplicates data. Auto Loader also adds a `_rescued_data` column that captures values
> that do not match the inferred schema.

## 1️⃣ Connect to Spark

In [1]:
from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-batch-ingestion")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 17:16:19.965 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-batch-ingestion | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_prod | env = prod


## 2️⃣ Peek at the raw sources

Spark can read the files directly from the Volume path — no copy, no staging.

### 🚕 Parquet — trips

In [2]:
SRC_PARQUET = f"/Volumes/{CATALOG}/raw_files/landing/parquet"
SRC_AVRO = f"/Volumes/{CATALOG}/raw_files/landing/avro"

source_trips = spark.read.parquet(SRC_PARQUET)
print(f"Parquet source: {source_trips.count():,} rows | {len(source_trips.columns)} columns")
display(source_trips.toPandas())

Parquet source: 4,000 rows | 22 columns


,id,user_id,driver_id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,...,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,390d4b18-668c-4e2e-bf38-4de0715f244c,899ca782-e323-4d1a-bc1b-dacc18e19331,f3a3c571-7476-4899-b5a3-adb3254a9493,2,2026-10-06 19:31:17,2026-10-06 20:01:17,6,8.63,2,N,...,2,34.33,0.0,0.5,5.15,0.00,1.0,44.73,2.5,1.25
1,3a916254-b611-474e-a63a-903bbaedbd7e,3139d32c-93cd-49bf-9c94-1cf0dc98d2c1,9fb61a63-2069-4ab0-a33d-babce6fff3d9,1,2026-10-03 00:36:53,2026-10-03 00:40:53,3,8.98,1,N,...,4,27.18,0.0,0.5,4.08,0.00,1.0,32.76,0.0,0.00
2,304fdce1-205a-4bd5-96be-53f9ea1c4e82,48ca7651-92f5-4f7b-8323-d342df6a8f93,ec1ce9d2-a346-4908-8ca5-ae8772bcbe19,2,2026-10-09 07:55:31,2026-10-09 08:04:31,6,29.92,1,N,...,1,106.77,0.0,0.5,16.02,0.00,1.0,126.79,2.5,0.00
3,58fe8e67-39a1-4427-b57a-eb37dc0d0c2f,78b2b549-3bdb-409e-acc2-16a01bbc91f7,4cc6382e-e3f4-4ca3-99f0-46a5d05668ae,2,2026-10-04 08:29:24,2026-10-04 08:41:24,5,2.46,4,N,...,1,12.27,1.0,0.5,0.00,0.00,1.0,17.27,2.5,0.00
4,d0f2f7bb-fc2b-476d-b982-a992e20f4724,8edddfcd-1e52-4770-bf89-7142fe716b14,e8d21bb3-366b-4d12-b204-dfa27257d8a5,1,2026-10-02 21:31:50,2026-10-02 21:59:50,3,25.03,3,N,...,1,94.86,2.5,0.5,14.23,12.06,1.0,127.65,2.5,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,028a6d0c-581e-454f-be44-d1059a5dfc78,3d114802-2702-478b-9f0f-da8d05379ff6,fff569c2-ee2b-48e4-93d1-622569a81137,2,2026-10-01 05:48:41,2026-10-01 06:21:41,2,9.18,1,N,...,3,30.23,0.0,0.5,4.53,0.00,1.0,38.76,2.5,0.00
3996,9a8152e5-805f-464d-a299-eaf85f358800,4fa03f26-f6f7-40cc-a9ec-8e49d1bdb8c0,4cc6382e-e3f4-4ca3-99f0-46a5d05668ae,2,2026-10-08 17:08:21,2026-10-08 17:25:21,5,4.97,5,Y,...,2,15.50,0.5,0.5,2.32,0.00,1.0,22.32,2.5,0.00
3997,2e414f12-d94d-4b6e-8e9f-06a24382a0f1,3c20592f-c04a-46c4-b3b6-3fe1d1843324,d40f46a4-356c-4dcc-9391-a5da253465a9,2,2026-10-02 11:57:20,2026-10-02 12:12:20,1,7.73,3,N,...,1,32.62,0.0,0.5,4.89,10.29,1.0,53.05,2.5,1.25
3998,af38414a-3ec5-48f9-954c-8c12f6ba6cb6,1d48a071-ab61-47b1-b93b-4c3220500494,ff9a1b80-5ced-4e2e-b17f-6920daaafe5c,2,2026-10-07 07:29:34,2026-10-07 07:57:34,2,14.55,4,Y,...,1,52.39,0.0,0.5,0.00,0.00,1.0,56.39,2.5,0.00


### 👤 Avro — users

In [3]:
source_users = spark.read.format("avro").load(SRC_AVRO)
print(f"Avro source: {source_users.count():,} rows | {len(source_users.columns)} columns")
display(source_users.toPandas())

Avro source: 1,000 rows | 5 columns


,id,full_name,email,registration_date,is_plus_subscriber
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Pamela Johnson,martinphillip@example.com,2021-05-12,True
1,23b8c1e9-3924-46de-beb1-3b9046685257,Jeanne Edwards,emily13@example.net,2022-01-21,True
2,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9,Wendy Valdez,michael96@example.net,2022-04-07,True
3,972a8469-1641-4f82-8b9d-2434e465e150,Travis White,mariahwhite@example.com,2021-02-03,True
4,17fc695a-07a0-4a6e-8822-e8f36c031199,Kaitlyn Foster,johnsonbenjamin@example.org,2020-11-07,True
...,...,...,...,...,...
995,d56ce8ea-1959-4b5a-a7a8-f6366a35df59,Keith Clements,morgansteven@example.com,2026-04-02,True
996,f065c817-687a-4850-9d9b-62317d45d8ef,Roberto Maldonado,ycardenas@example.org,2020-04-24,True
997,b0b63bcf-0860-4833-879d-0cdaf396ea37,Elizabeth Robertson,brooke88@example.net,2022-12-17,True
998,71d79665-7181-4dcf-b79e-fc6e5edb0d3c,Dr. Sarah Brown DDS,murraykirsten@example.com,2021-07-07,False


## 3️⃣ Ingest into Bronze

Two services, one pattern. Each call blocks until the available files are fully processed.

```text
 landing/avro      ──▶ UserBatchIngestionService      ──▶ bronze.users
 landing/parquet   ──▶ TaxiTripBatchIngestionService  ──▶ bronze.taxi
```

In [4]:
from src.core.etl.bronze import (
    TaxiTripBatchIngestionService,
    UserBatchIngestionService,
)

user_service = UserBatchIngestionService()
user_service.run(spark)

taxi_service = TaxiTripBatchIngestionService()
taxi_service.run(spark)

print("✅ Batch ingestion finished")

2026-10-09 17:16:28.430 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_prod.bronze.users
2026-10-09 17:16:28.431 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [avro] from: /Volumes/yandex_go_prod/raw_files/landing/avro
2026-10-09 17:16:28.595 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_prod.bronze.users
2026-10-09 17:16:42.296 | INFO     | src.core.etl.bronze.base:load:111 - Successfully loaded in yandex_go_prod.bronze.users
2026-10-09 17:16:42.297 | INFO     | src.core.etl.bronze.base:run:123 - Start data ingestion for yandex_go_prod.bronze.taxi
2026-10-09 17:16:42.298 | INFO     | src.core.etl.bronze.base:extract:63 - Reading data [parquet] from: /Volumes/yandex_go_prod/raw_files/landing/parquet
2026-10-09 17:16:42.298 | INFO     | src.core.etl.bronze.base:load:99 - Writing data in table Delta: yandex_go_prod.bronze.taxi
2026-10-09 17:16:56.877 | INFO     | src.core.etl.bronze.base:load:1

✅ Batch ingestion finished


---

## ✅ Checkpoint

- [x] Parquet and Avro files read directly from the Volume
- [x] `bronze.taxi` and `bronze.users` written as Delta tables
- [x] Row counts match the sources; re-running adds nothing

> **Next up ▶ [3 · Streaming Ingestion & Delta Features](./3.%20Streaming%20Ingestion%20%26%20Delta%20Features.ipynb)** — stream drivers into Bronze, evolve the schema and travel through time.

<div align="center"><sub>Yandex Go Analytics · notebook 2 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>